# Adding & Removing Rows and Columns in Pandas

You will learn:

- Adding a column with `df.loc[:, "new"] = ...`
- Adding a column from a calculation on another column
- Adding a row with `df.loc[label] = {...}`
- Changing values with `df.loc[condition, columns] = [...]`
- Why `df[condition][cols] = ...` does **not** work (SettingWithCopyWarning)
- Removing rows and columns with `drop()`
- When `inplace=True` is needed

In [1]:
import pandas as pd

In [2]:
# Sample Titanic-like data
data = {
    "Name": ["Jack", "Rose", "Tom", "Ananda"],
    "Age": [22, 19, 25, 20],
    "Sex": ["male", "female", "male", "male"],
    "Survived": [1, 1, 0, 1],
    "Fare": [50, 100, 30, 60]
}

df = pd.DataFrame(data)
df


,Name,Age,Sex,Survived,Fare
0,Jack,22,male,1,50
1,Rose,19,female,1,100
2,Tom,25,male,0,30
3,Ananda,20,male,1,60


In [3]:
# Add a new column with default value
# Using .loc to add column
df.loc[:, "Embarked"] = "C"
df


,Name,Age,Sex,Survived,Fare,Embarked
0,Jack,22,male,1,50,C
1,Rose,19,female,1,100,C
2,Tom,25,male,0,30,C
3,Ananda,20,male,1,60,C


In [4]:
# Add a column based on calculation
df.loc[:, "Fare_after_tax"] = df["Fare"] * 1.10
df

,Name,Age,Sex,Survived,Fare,Embarked,Fare_after_tax
0,Jack,22,male,1,50,C,55.0
1,Rose,19,female,1,100,C,110.0
2,Tom,25,male,0,30,C,33.0
3,Ananda,20,male,1,60,C,66.0


In [5]:
# add another columns
df.loc[:,"Age increase"]=df["Age"]+10
df

,Name,Age,Sex,Survived,Fare,Embarked,Fare_after_tax,Age increase
0,Jack,22,male,1,50,C,55.0,32
1,Rose,19,female,1,100,C,110.0,29
2,Tom,25,male,0,30,C,33.0,35
3,Ananda,20,male,1,60,C,66.0,30


In [6]:
# CAREFUL — a chained assignment like
#     df.loc[:, "Age decrease"] = df["Age"] = 10
# does NOT subtract anything. It assigns 10 to df["Age"] first and then copies
# that 10 into the new column, so the real ages are lost.
# What was meant:
df.loc[:, "Age decrease"] = df["Age"] - 10
df

,Name,Age,Sex,Survived,Fare,Embarked,Fare_after_tax,Age increase,Age decrease
0,Jack,22,male,1,50,C,55.0,32,12
1,Rose,19,female,1,100,C,110.0,29,9
2,Tom,25,male,0,30,C,33.0,35,15
3,Ananda,20,male,1,60,C,66.0,30,10


In [7]:
# Keep only selected columns using .loc
df=df.loc[:, ["Name", "Age", "Sex", "Survived", "Fare"]]



In [8]:
df

,Name,Age,Sex,Survived,Fare
0,Jack,22,male,1,50
1,Rose,19,female,1,100
2,Tom,25,male,0,30
3,Ananda,20,male,1,60


In [9]:
df.loc[5] = {"Name": "Leo", "Age": 30, "Sex": "male"}
# add new rows using the dictionary

In [10]:
df

,Name,Age,Sex,Survived,Fare
0,Jack,22,male,1.0,50.0
1,Rose,19,female,1.0,100.0
2,Tom,25,male,0.0,30.0
3,Ananda,20,male,1.0,60.0
5,Leo,30,male,NaN,NaN


for dictionary
Pandas will automatically fill missing columns with NaN.


Fetch Multiple Rows at the same time

In [11]:
df.loc[0:2]

,Name,Age,Sex,Survived,Fare
0,Jack,22,male,1.0,50.0
1,Rose,19,female,1.0,100.0
2,Tom,25,male,0.0,30.0


Fetch Multiple Columns at the same time

In [12]:
df.loc[:, ["Name", "Age", "Fare"]]

,Name,Age,Fare
0,Jack,22,50.0
1,Rose,19,100.0
2,Tom,25,30.0
3,Ananda,20,60.0
5,Leo,30,NaN


In [13]:
import pandas as pd

# Sample DataFrame
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Clara'],
    'Age': [25, 30, 28],
    'Sex': ['female', 'male', 'female']
})
df

,Name,Age,Sex
0,Alice,25,female
1,Bob,30,male
2,Clara,28,female


In [14]:
# Change Bob's Age and Sex
df.loc[df['Name'] == 'Bob', ['Age', 'Sex']] = [31, 'male']

df


,Name,Age,Sex
0,Alice,25,female
1,Bob,31,male
2,Clara,28,female


In [15]:
# Task: now change the name and age of Clara
df.loc[df['Name'] == 'Clara', ['Name', 'Age']] = ['Antonella', 50]
df

,Name,Age,Sex
0,Alice,25,female
1,Bob,31,male
2,Antonella,50,female


In [16]:
# Changes multiple columns at once.
# 'Status' does not exist yet — .loc creates it and leaves NaN in the other rows.
df.loc[df['Name'] == 'Antonella', ['Age', 'Sex', 'Status']] = [29, 'female', 'Married']

df

,Name,Age,Sex,Status
0,Alice,25,female,NaN
1,Bob,31,male,NaN
2,Antonella,29,female,Married


In [17]:
df.loc[:, ["Height", "Weight"]] =  [5.2, 65]
df

,Name,Age,Sex,Status,Height,Weight
0,Alice,25,female,NaN,5.2,65
1,Bob,31,male,NaN,5.2,65
2,Antonella,29,female,Married,5.2,65


how to add different hieght and weight for each person

In [18]:
df.loc[0, ["Height", "Weight"]] = [5.4, 55]
df.loc[1, ["Height", "Weight"]] = [5.9, 72]
df.loc[2, ["Height", "Weight"]] = [5.2, 60]

df

,Name,Age,Sex,Status,Height,Weight
0,Alice,25,female,NaN,5.4,55
1,Bob,31,male,NaN,5.9,72
2,Antonella,29,female,Married,5.2,60


### Do **not** do it like this

The cell below uses **chained indexing**: `df[mask]` builds a temporary object
first, and the assignment lands in that temporary object instead of in `df`.
Pandas warns with `SettingWithCopyWarning`, and `df` is left unchanged.

Always write `df.loc[condition, columns] = value` instead.

In [19]:
# This raises SettingWithCopyWarning and does NOT change df — that is the point
df[df['Name'] == 'Antonella'][['Age', 'Sex', 'Status']] = [32, 'female', 'Married']

/tmp/ipykernel_1253/3745157405.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df['Name'] == 'Antonella'][['Age', 'Sex', 'Status']] = [32, 'female', 'Married']


In [20]:
# Proof: Age is still 29, the write went nowhere
df

,Name,Age,Sex,Status,Height,Weight
0,Alice,25,female,NaN,5.4,55
1,Bob,31,male,NaN,5.9,72
2,Antonella,29,female,Married,5.2,60


In [21]:
import pandas as pd

# Sample DataFrame
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Clara'],
    'Age': [25, 30, 28]
})
df


,Name,Age
0,Alice,25
1,Bob,30
2,Clara,28


In [22]:
# Add a new column 'Sex'
df.loc[:, 'Sex'] = ['female', 'male', 'female']

df

,Name,Age,Sex
0,Alice,25,female
1,Bob,30,male
2,Clara,28,female


In [23]:
df.set_index("Age",inplace=True)

In [24]:
df

,Name,Sex
Age,,
25,Alice,female
30,Bob,male
28,Clara,female


In [25]:
df.loc[25,"Sex"]

'female'

### Examples of when you need `inplace=True` (including index operations)

| Operation | Default behavior | With `inplace=True` |
|-----------|----------------|------------------|
| `df.drop("x", axis=1)` | Returns a new DataFrame without column `"x"`; original `df` stays the same | Original `df` loses column `"x"` |
| `df.drop(0, axis=0)` | Returns a new DataFrame without row 0; original `df` unchanged | Original `df` loses row 0 |
| `df.sort_values("col")` | Returns a sorted copy; original `df` unchanged | Original `df` is sorted |
| `df.sort_index()` | Returns a copy sorted by index; original `df` unchanged | Original `df` is sorted by index |
| `df.fillna(0)` | Returns a copy with NaNs replaced | Original `df` modified |
| `df.rename(columns={"old":"new"})` | Returns a copy with renamed columns | Original `df` modified |
| `df.rename(index={0:"zero"})` | Returns a copy with renamed index | Original `df` index modified |
| `df.set_index("col")` | Returns a new DataFrame with `"col"` as index; original df unchanged | Original `df` now uses `"col"` as index |
| `df.reset_index()` | Returns a copy with index reset; original df unchanged | Original `df` index reset |


---

## Removing rows and columns with `drop()`

`drop()` is the counterpart of everything above.

- `df.drop(columns=[...])` removes **columns**
- `df.drop(index=[...])` removes **rows** (by their label)
- like most pandas methods it returns a **copy**, so either reassign or pass `inplace=True`

In [26]:
import pandas as pd

people = pd.DataFrame({
    "Name":   ["Alice", "Bob", "Clara", "Dipesh", "Elina"],
    "Age":    [25, 30, 28, 41, 36],
    "City":   ["Kathmandu", "Pokhara", "Butwal", "Dharan", "Chitwan"],
    "Temp":   [1, 1, 1, 1, 1],
    "Salary": [50000, 62000, 48000, 90000, 71000]
})
people

,Name,Age,City,Temp,Salary
0,Alice,25,Kathmandu,1,50000
1,Bob,30,Pokhara,1,62000
2,Clara,28,Butwal,1,48000
3,Dipesh,41,Dharan,1,90000
4,Elina,36,Chitwan,1,71000


In [27]:
# Remove ONE column
people.drop(columns=["Temp"])

,Name,Age,City,Salary
0,Alice,25,Kathmandu,50000
1,Bob,30,Pokhara,62000
2,Clara,28,Butwal,48000
3,Dipesh,41,Dharan,90000
4,Elina,36,Chitwan,71000


In [28]:
# ...but the original still has it, because drop returned a copy
people.columns

Index(['Name', 'Age', 'City', 'Temp', 'Salary'], dtype='object')

In [29]:
# Keep the change: reassign (preferred) or use inplace=True
people = people.drop(columns=["Temp"])
people

,Name,Age,City,Salary
0,Alice,25,Kathmandu,50000
1,Bob,30,Pokhara,62000
2,Clara,28,Butwal,48000
3,Dipesh,41,Dharan,90000
4,Elina,36,Chitwan,71000


In [30]:
# Remove SEVERAL columns at once
people.drop(columns=["City", "Salary"])

,Name,Age
0,Alice,25
1,Bob,30
2,Clara,28
3,Dipesh,41
4,Elina,36


In [31]:
# Remove rows by their index LABEL
people.drop(index=[0, 2])

,Name,Age,City,Salary
1,Bob,30,Pokhara,62000
3,Dipesh,41,Dharan,90000
4,Elina,36,Chitwan,71000


In [32]:
# Remove the rows that match a condition.
# There is no "drop where" — you select the labels first, then drop them.
young = people.loc[people["Age"] < 30].index
people.drop(index=young)

,Name,Age,City,Salary
1,Bob,30,Pokhara,62000
3,Dipesh,41,Dharan,90000
4,Elina,36,Chitwan,71000


In [33]:
# The same thing written as a filter — usually clearer, and it is what you
# will use most of the time
people.loc[people["Age"] >= 30]

,Name,Age,City,Salary
1,Bob,30,Pokhara,62000
3,Dipesh,41,Dharan,90000
4,Elina,36,Chitwan,71000


In [34]:
# axis= is the older style: axis=1 means columns, axis=0 means rows
print(people.drop("Salary", axis=1).columns.tolist())
print(people.drop(0, axis=0).index.tolist())

['Name', 'Age', 'City']
[1, 2, 3, 4]


In [35]:
# drop_duplicates removes repeated ROWS
dupes = pd.DataFrame({"Name": ["A", "B", "A", "C"], "Score": [1, 2, 1, 3]})
print("before:", len(dupes))
print("after :", len(dupes.drop_duplicates()))
dupes.drop_duplicates()

before: 4
after : 3


,Name,Score
0,A,1
1,B,2
3,C,3


In [36]:
# dropna removes rows that have missing values
gaps = pd.DataFrame({"Name": ["A", "B", "C"], "Score": [1, None, 3]})
print(gaps)
print()
print(gaps.dropna())

  Name  Score
0    A    1.0
1    B    NaN
2    C    3.0

  Name  Score
0    A    1.0
2    C    3.0


### Key takeaways

| Goal | Code |
|------|------|
| Add a column | `df.loc[:, "new"] = value` |
| Add a calculated column | `df.loc[:, "new"] = df["a"] * 2` |
| Add a row | `df.loc[label] = {...}` |
| Change values safely | `df.loc[condition, cols] = [...]` |
| Remove columns | `df.drop(columns=["a", "b"])` |
| Remove rows by label | `df.drop(index=[0, 2])` |
| Remove rows by condition | `df.loc[~condition]` (filtering is clearer than dropping) |
| Remove repeated rows | `df.drop_duplicates()` |
| Remove rows with gaps | `df.dropna()` |

**Remember:** `drop()` returns a copy. Reassign it or pass `inplace=True`, otherwise
nothing changes.